In [ ]:
import os

# Pega o diretório pai de 'scripts'
try:
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    BASE_DIR = os.path.dirname(os.getcwd())

raw_folder = os.path.join(BASE_DIR, "raw")
processed_folder = os.path.join(BASE_DIR, "processed")

In [ ]:
import os
import pandas as pd

def preprocess_all_files(raw_folder='./raw', processed_folder='./processed', use_adj_close=True):
    os.makedirs(processed_folder, exist_ok=True)

    for file_name in os.listdir(raw_folder):
        if not file_name.endswith('.csv'):
            continue

        file_path = os.path.join(raw_folder, file_name)
        print(f"🔹 Processando: {file_name}")

        try:
            # Lê as duas primeiras linhas pra detectar se existe linha extra (metadados)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                first_lines = [next(f).strip() for _ in range(3)]
        except Exception:
            first_lines = []

        # Se a segunda linha parece ser metadado (ex: começa com "Ticker,"), pulamos apenas ela
        skiprows = []
        if len(first_lines) >= 2 and first_lines[1].lower().startswith('ticker'):
            skiprows = [1]  # mantém a primeira linha como header
        # Caso seu arquivo realmente tenha dois cabeçalhos estranhos, adapte skiprows = [0,1]

        # Leitura do CSV com skiprows condicional
        df = pd.read_csv(file_path, skiprows=skiprows, engine='python')

        # Normaliza nomes de colunas (tira espaços e deixa minúsculas)
        df.columns = df.columns.str.strip()

        # Se a primeira coluna estiver como 'Price', renomeia para 'Date'
        # (muitas exportações usam 'Price' para a coluna de datas)
        first_col = df.columns[0]
        if first_col.lower() in ['price', 'date', 'data']:
            df = df.rename(columns={first_col: 'Date'})
        else:
            # tenta forçar Date se possível (se a coluna contém valores no formato de data)
            try:
                pd.to_datetime(df[first_col])
                df = df.rename(columns={first_col: 'Date'})
            except Exception:
                pass  # fica como está e tratamento mais abaixo pode falhar com aviso

        # garante que existem colunas de preço; escolhe adj close quando disponível
        col_candidates = {c.lower(): c for c in df.columns}
        price_col = None
        if use_adj_close and 'adj close' in col_candidates:
            price_col = col_candidates['adj close']
        elif 'close' in col_candidates:
            price_col = col_candidates['close']
        else:
            raise KeyError(f"Nenhuma coluna 'Adj Close' ou 'Close' encontrada em {file_name}. Cols: {list(df.columns)}")

        # seleciona e padroniza
        df = df[['Date', price_col]].rename(columns={price_col: 'Close'})

        # tipos e limpeza
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
        df = df.dropna(subset=['Date', 'Close']).drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)

        output_path = os.path.join(processed_folder, file_name)
        df.to_csv(output_path, index=False)
        print(f"Salvo: {output_path}\n")


In [ ]:
preprocess_all_files(raw_folder=raw_folder, processed_folder=processed_folder, use_adj_close=True)